In [9]:
from pathlib import Path
import pandas as pd

# Set folder paths
raw_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\instagram\instagram_comments")
cleaned_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\instagram\instagram_comments_cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True)

# List all raw files (CSV and Excel)
raw_files = sorted([f for f in raw_folder.iterdir() if f.suffix in ['.csv', '.xlsx']])
cleaned_files = {f.stem.replace('_cleaned', '') for f in cleaned_folder.glob("*_cleaned.csv")}

# Get next unprocessed file
next_file = None
for file in raw_files:
    name = file.stem
    if name not in cleaned_files:
        next_file = file
        break

print(f"👉 Next file to clean: {next_file.name}" if next_file else "✅ All files cleaned.")



👉 Next file to clean: instagram (12).xlsx


In [11]:
input_path = next_file

if input_path.suffix == '.csv':
    df = pd.read_csv(input_path, dtype=str, encoding='utf-8', engine='python', on_bad_lines='skip')
elif input_path.suffix == '.xlsx':
    df = pd.read_excel(input_path, dtype=str, engine='openpyxl')
else:
    raise ValueError(f"🛑 Unsupported file type: {input_path.suffix}")

print(f"\n📂 Loaded: {input_path.name}")
print("🧾 Columns:", df.columns.tolist())
df.head()




📂 Loaded: instagram (12).xlsx
🧾 Columns: ['x1i10hfl href', 'xpdipgo src', 'x1i10hfl', 'x1i10hfl href 2', '_ap3a', 'x1i10hfl href 3', '_a9ze', 'x193iq5w', 'x193iq5w 2']


,x1i10hfl href,xpdipgo src,x1i10hfl,x1i10hfl href 2,_ap3a,x1i10hfl href 3,_a9ze,x193iq5w,x193iq5w 2
0,https://www.instagram.com/kaklaw/,https://instagram.flos5-3.fna.fbcdn.net/v/t51....,kaklaw,https://www.instagram.com/kaklaw/,So many precious coins are spent before this b...,https://www.instagram.com/p/DKe8mpJoOnC/c/1807...,4w,370 likes,Reply
1,https://www.instagram.com/hanzala_mazhar_49/,https://instagram.flos5-1.fna.fbcdn.net/v/t51....,hanzala_mazhar_49,https://www.instagram.com/hanzala_mazhar_49/,Human being is a collection of infinite desire...,https://www.instagram.com/p/DKe8mpJoOnC/c/1838...,4w,196 likes,Reply
2,https://www.instagram.com/maria1991rita/,https://instagram.fbgw40-1.fna.fbcdn.net/v/t51...,maria1991rita,https://www.instagram.com/maria1991rita/,This hurts. It feels like all of my figs are w...,https://www.instagram.com/p/DKe8mpJoOnC/c/1788...,4w,37 likes,Reply
3,https://www.instagram.com/carl_dobsky/,https://instagram.flos5-1.fna.fbcdn.net/v/t51....,carl_dobsky,https://www.instagram.com/carl_dobsky/,Absolutely. But figuring out what matters is h...,https://www.instagram.com/p/DKe8mpJoOnC/c/1795...,4w,76 likes,Reply
4,https://www.instagram.com/_hanantaha/,https://instagram.flos5-3.fna.fbcdn.net/v/t51....,_hanantaha,https://www.instagram.com/_hanantaha/,How wonderful! I'd love to follow... is this a...,https://www.instagram.com/p/DKe8mpJoOnC/c/1793...,4w,11 likes,Reply


In [12]:
# 🧼 Step 1.5: Strip any whitespace in column headers
df.columns = df.columns.str.strip()

# ✅ Step 2: Map known columns to standard structure
column_map = {
    'x1i10hfl': 'username',
    '_ap3a': 'comment',
    'x1i10hfl href 3': 'comment_url',
    'x193iq5w': 'time_posted',
    'x1i10hfl href': 'profile_url',  # optional
}

# 🧠 Try matching only known columns
existing = [col for col in column_map if col in df.columns]

if not existing:
    raise ValueError(f"🛑 No known columns detected. Columns found: {df.columns.tolist()}")

# 🧾 Subset and rename
df = df[existing].rename(columns={col: column_map[col] for col in existing})

print("✅ Columns after renaming:", df.columns.tolist())
df.head()


✅ Columns after renaming: ['username', 'comment', 'comment_url', 'time_posted', 'profile_url']


,username,comment,comment_url,time_posted,profile_url
0,kaklaw,So many precious coins are spent before this b...,https://www.instagram.com/p/DKe8mpJoOnC/c/1807...,370 likes,https://www.instagram.com/kaklaw/
1,hanzala_mazhar_49,Human being is a collection of infinite desire...,https://www.instagram.com/p/DKe8mpJoOnC/c/1838...,196 likes,https://www.instagram.com/hanzala_mazhar_49/
2,maria1991rita,This hurts. It feels like all of my figs are w...,https://www.instagram.com/p/DKe8mpJoOnC/c/1788...,37 likes,https://www.instagram.com/maria1991rita/
3,carl_dobsky,Absolutely. But figuring out what matters is h...,https://www.instagram.com/p/DKe8mpJoOnC/c/1795...,76 likes,https://www.instagram.com/carl_dobsky/
4,_hanantaha,How wonderful! I'd love to follow... is this a...,https://www.instagram.com/p/DKe8mpJoOnC/c/1793...,11 likes,https://www.instagram.com/_hanantaha/


In [13]:
expected_order = [
    'username',
    'video_id',
    'comment_url',
    'time_posted',
    'time_posted_parsed',
    'comment',
    'comment_original',
    'like_count',
    'is_duplicate_comment'
]

cleaned_df = cleaned_df[expected_order]


In [14]:
output_path = cleaned_folder / f"{input_path.stem}_cleaned.csv"
cleaned_df.to_csv(output_path, index=False)

print(f"✅ Cleaned and saved: {output_path.name}")


✅ Cleaned and saved: instagram (12)_cleaned.csv


##### Bulk clean all of the remaining files

In [6]:
from pathlib import Path
import pandas as pd

# Set folder paths
raw_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\youtube\youtube_comments")
cleaned_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\youtube\youtube_comments_cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True)

# List raw and cleaned files
raw_files = sorted([f for f in raw_folder.glob("*.csv")])
cleaned_files = {f.stem.replace('_cleaned', '') for f in cleaned_folder.glob("*_cleaned.csv")}

# Track failed files
failed_files = []

# Loop through all uncleaned files
for file in raw_files:
    if file.stem in cleaned_files:
        continue

    try:
        print(f"\n🔄 Cleaning: {file.name}")
        df = pd.read_csv(file, dtype=str, encoding='utf-8', engine='python')

        def find_col(substring, exclude=None):
            for col in df.columns:
                if substring.lower() in col.lower():
                    if exclude and exclude.lower() in col.lower():
                        continue
                    return col
            return None

        username_col = find_col('yt-simple-endpoint', exclude='href')
        comment_url_col = next((c for c in df.columns if 'href' in c and 'watch' in (df[c].dropna().iloc[0] if not df[c].dropna().empty else '')), None)
        time_posted_col = find_col('style-scope')
        comment_text_col = find_col('yt-core-attributed-string')
        like_count_col = find_col('style-scope')

        # Overrides if necessary
        if 'style-scope 3' in df.columns:
            like_count_col = 'style-scope 3'
        if 'yt-core-attributed-string 3' in df.columns and not comment_text_col:
            comment_text_col = 'yt-core-attributed-string 3'

        video_id_series = None
        if comment_url_col:
            video_id_series = df[comment_url_col].str.extract(r'v=([A-Za-z0-9_-]+)')[0]

        cleaned_df = pd.DataFrame({
            'username': df[username_col] if username_col else None,
            'video_id': video_id_series,
            'comment_url': df[comment_url_col] if comment_url_col else None,
            'time_posted': df[time_posted_col] if time_posted_col else None,
            'time_posted_parsed': '',
            'comment': df[comment_text_col] if comment_text_col else None,
            'comment_original': df[comment_text_col] if comment_text_col else None,
            'like_count': df[like_count_col] if like_count_col else None,
            'is_duplicate_comment': ''
        })

        def normalize_likes(x):
            if isinstance(x, str):
                x = x.strip().lower().replace(',', '')
                if x.endswith('k'):
                    return int(float(x[:-1]) * 1000)
                if x.endswith('m'):
                    return int(float(x[:-1]) * 1_000_000)
                if x.isdigit():
                    return int(x)
            return x

        cleaned_df['like_count'] = cleaned_df['like_count'].apply(normalize_likes)

        expected_order = [
            'username', 'video_id', 'comment_url', 'time_posted',
            'time_posted_parsed', 'comment', 'comment_original',
            'like_count', 'is_duplicate_comment'
        ]

        cleaned_df = cleaned_df[expected_order]

        output_path = cleaned_folder / f"{file.stem}_cleaned.csv"
        cleaned_df.to_csv(output_path, index=False)
        print(f"✅ Saved: {output_path.name}")

    except Exception as e:
        print(f"❌ Failed: {file.name} → {e}")
        failed_files.append(file.name)

# Summary
if failed_files:
    print("\n❗ The following files could not be cleaned:")
    for f in failed_files:
        print(f"- {f}")
else:
    print("\n🎉 All files cleaned successfully!")



🔄 Cleaning: youtube (29).csv
✅ Saved: youtube (29)_cleaned.csv

🔄 Cleaning: youtube (3).csv
✅ Saved: youtube (3)_cleaned.csv

🔄 Cleaning: youtube (30).csv
✅ Saved: youtube (30)_cleaned.csv

🔄 Cleaning: youtube (31).csv
✅ Saved: youtube (31)_cleaned.csv

🔄 Cleaning: youtube (32).csv
✅ Saved: youtube (32)_cleaned.csv

🔄 Cleaning: youtube (33).csv
✅ Saved: youtube (33)_cleaned.csv

🔄 Cleaning: youtube (34).csv
✅ Saved: youtube (34)_cleaned.csv

🔄 Cleaning: youtube (35).csv
✅ Saved: youtube (35)_cleaned.csv

🔄 Cleaning: youtube (36).csv
✅ Saved: youtube (36)_cleaned.csv

🔄 Cleaning: youtube (37).csv
✅ Saved: youtube (37)_cleaned.csv

🔄 Cleaning: youtube (38).csv
✅ Saved: youtube (38)_cleaned.csv

🔄 Cleaning: youtube (39).csv
✅ Saved: youtube (39)_cleaned.csv

🔄 Cleaning: youtube (4).csv
✅ Saved: youtube (4)_cleaned.csv

🔄 Cleaning: youtube (40).csv
✅ Saved: youtube (40)_cleaned.csv

🔄 Cleaning: youtube (41).csv
✅ Saved: youtube (41)_cleaned.csv

🔄 Cleaning: youtube (42).csv
✅ Saved: youtu

#### Let's merge them all in to one file

In [7]:
from pathlib import Path
import pandas as pd

# Folder containing all cleaned files
cleaned_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\youtube\youtube_comments_cleaned")

# List all cleaned CSV files
cleaned_files = sorted(cleaned_folder.glob("*_cleaned.csv"))

# Read and concatenate all cleaned DataFrames
all_dfs = []
for file in cleaned_files:
    try:
        df = pd.read_csv(file, dtype=str)
        df['source_file'] = file.name  # Optional: keep track of which file it came from
        all_dfs.append(df)
    except Exception as e:
        print(f"❌ Could not read {file.name}: {e}")

# Merge into one big DataFrame
merged_df = pd.concat(all_dfs, ignore_index=True)
print(f"✅ Merged {len(cleaned_files)} files → Final shape: {merged_df.shape}")

# Save the merged file
output_path = cleaned_folder / "all_youtube_comments_cleaned.csv"
merged_df.to_csv(output_path, index=False)
print(f"💾 Saved merged file to: {output_path}")


✅ Merged 225 files → Final shape: (62453, 10)
💾 Saved merged file to: C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\youtube\youtube_comments_cleaned\all_youtube_comments_cleaned.csv
